# KneeMIL: submit to the competition from Colab

Builds the Kaggle submit kernel from any mix of runs, waits for it, checks that it really wrote
`submission.csv`, and submits it.

* No GPU is needed in Colab: the kernel runs on Kaggle.
* Building the kernel costs ~15 min of Kaggle GPU quota. Kaggle's re-run on the hidden test is free.

**One time:** add `KAGGLE_API_TOKEN` and `KAGGLE_USERNAME` (`danielaijdkx`) in Colab's 🔑 *Secrets*
panel. Never paste the token into a cell.

Edit the configuration cell, then *Runtime → Run all*.

In [ ]:
# ---- what to submit
TRAIN_KERNELS  = ["kneemri-train-r0", "kneemri-train-r1", "kneemri-train-r2",
                  "kneemri-train-r3", "kneemri-train-r5", "kneemri-train-r6"]   # runs trained on Kaggle
TRAIN_DATASETS = []          # runs trained in Colab, e.g. ["danielaijdkx/kneemri-r7"]
MESSAGE        = "own r0-r6 rank mean"
BRANCH         = "claude/nifty-cannon-ivkdzl"

In [ ]:
import os, sys, subprocess, time, json
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import userdata
    for k in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME"):
        os.environ[k] = userdata.get(k)
assert os.environ.get("KAGGLE_API_TOKEN") and os.environ.get("KAGGLE_USERNAME"), "set the two secrets first"

def sh(cmd, check=True, capture=False):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if check and r.returncode:
        print(r.stdout if capture else "", r.stderr if capture else "")
        raise RuntimeError(f"failed ({r.returncode}): {cmd}")
    return r.stdout if capture else r.returncode

sh("pip install -q -U kaggle")
CODE = Path("/content/RSNA") if Path("/content").exists() else Path.home() / "RSNA"
if not CODE.exists():
    sh(f"git clone -q -b {BRANCH} https://github.com/Daniel766hi/RSNA {CODE}")
else:
    sh(f"git -C {CODE} pull -q")
os.chdir(CODE)
print(sh("kaggle quota", capture=True))

In [ ]:
# ---- build the submit kernel (attaches the runs; the kernel installs codecs offline and predicts)
cmd = "python kaggle/orchestrate.py push submit --train " + " ".join(TRAIN_KERNELS)
if TRAIN_DATASETS:
    cmd += " --train-datasets " + " ".join(TRAIN_DATASETS)
sh(cmd)

In [ ]:
# ---- wait for the kernel version to finish (~5-20 min)
user = os.environ["KAGGLE_USERNAME"]
time.sleep(60)
while True:
    st = sh(f"kaggle kernels status {user}/kneemri-submit", capture=True, check=False).lower()
    print(time.strftime("%H:%M:%S"), st.strip(), flush=True)
    if any(s in st for s in ("complete", "error", "cancel")):
        break
    time.sleep(60)
assert "complete" in st, "kernel failed: open https://www.kaggle.com/code/%s/kneemri-submit to read the log" % user

In [ ]:
# ---- check the output and submit (refuses a missing or all-0.5 fallback submission)
sh(f'python kaggle/orchestrate.py submit kneemri-submit -m "{MESSAGE}"')
time.sleep(30)
print(sh("kaggle competitions submissions rsna-knee-abnormality-detection", capture=True).splitlines()[:6])
print("The public score appears after Kaggle re-runs the kernel on the hidden test (1-3 h). Re-run the cell below to refresh.")

In [ ]:
print("\n".join(sh("kaggle competitions submissions rsna-knee-abnormality-detection", capture=True).splitlines()[:8]))